In [ ]:
import os
from collections import OrderedDict

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import matplotlib.pyplot as plt
import numpy as np
import cv2

from PIL import Image


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ========================================
# IMAGE CONFIG
# ========================================
#IMAGE_PATH = "../../results/Shampoo_NOBGR_pix2pix_StructCond_V1_Stage23_COMPLETESyn/test_latest/images_fake/deb1f5bb-2026-03-03_15-40-15-784_te_000095_fake_B.png"
IMAGE_PATH = "../../results/Shampoo_NOBGR_pix2pix_StructCond_V1_Stage23_COMPLETESyn/test_latest/images_fake/9e487d1b-2026-03-03_16-54-56-681_te_000063_fake_B.png"



#IMAGE_PATH = "../../data/interim/Stage1/gray_clahe_1500x1000_noborder_aug/2026-01-21_10-38-24-999_aug01.png"
#IMAGE_PATH = "../../data/interim/Stage2/gray_clahe_1500x1000_noborder_aug/2026-01-21_15-46-35-205_aug01.png"
#IMAGE_PATH = "../../data/interim/Stage2/gray_clahe_1500x1000_noborder_aug/2026-01-21_15-59-19-201_aug01.png"
#IMAGE_PATH = "../../data/interim/Stage2/color_clahe_1500x1000_noborder_aug/2026-01-21_15-46-35-205_aug01.png"
#MAGE_PATH = "../../data/interim/Stage1/color_clahe_1500x1000_noborder_aug/2026-01-21_10-38-24-999_aug01.png"

#IMAGE_PATH = "../../data/raw/SHAMPOOBLADEWITHTRAY_COMPLETE/3d5018a3-2026-03-03_16-47-04-096.png"
#IMAGE_PATH = "../../data/raw/SHAMPOOBLADEWITHTRAY_COMPLETE/0c52233c-2026-03-03_16-03-48-194.png"
#IMAGE_PATH = "../../results/Shampoo_NOBGR_pix2pix_StructCond_V1_Stage23_COMPLETESyn/test_latest/images_real/5a5781af-2026-03-03_16-44-40-914_te_000061_real_B.png"


IMAGE_SIZE = 256
IMAGE_MODE = "gray"

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

GRAY_MEAN = (0.5,)
GRAY_STD  = (0.25,)

if IMAGE_MODE == "rgb":
    transform = T.Compose([
        T.Resize(int(IMAGE_SIZE * 1.10)),
        T.CenterCrop(IMAGE_SIZE),
        T.ToTensor(),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
else:
    transform = T.Compose([
        T.Grayscale(num_output_channels=1),
        T.Resize(int(IMAGE_SIZE * 1.10)),
        T.CenterCrop(IMAGE_SIZE),
        T.ToTensor(),
        T.Normalize(GRAY_MEAN, GRAY_STD),
    ])


# ========================================
# MODELS
# ========================================
class SimpleCNN_GAP1(nn.Module):
    def __init__(self, in_channels=1, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Linear(64, 64), nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


class SimpleCNN_GAP2(nn.Module):
    def __init__(self, in_channels=1, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(256, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Linear(512, 512), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)
"""    
MODEL_SPECS = [
    {
        "name": "Stage1",
        "ckpt": "../../CNN_models/classifier/Stage1/Stage1.pt",
        "build": lambda: SimpleCNN_GAP1(in_channels=3, num_classes=2),
        "class_names": {0: "Empty", 1: "Non-Empty"},
    },
    {
        "name": "Stage2",
        "ckpt": "../../CNN_models/classifier/Stage2/Stage2.pt",
        "build": lambda: SimpleCNN_GAP2(in_channels=3, num_classes=2),
        "class_names": {0: "Overlap", 1: "Isolated"},
    },
    {
        "name": "Stage3",
        "ckpt": "../../CNN_models/classifier/Stage3/Stage3.pt",
        "build": lambda: SimpleCNN_GAP2(in_channels=3, num_classes=2),
        "class_names": {0: "Cluttered", 1: "Not-cluttered"},
    },
]
"""

MODEL_SPECS = [
    {
        "name": "Stage1",
        "ckpt": "../../CNN_models/classifier/Stage1/Stage1_grayV2.pt",
        "build": lambda: SimpleCNN_GAP1(in_channels=1, num_classes=2),
        "class_names": {0: "Empty", 1: "Non-Empty"},
    },
    {
        "name": "Stage2",
        "ckpt": "../../CNN_models/classifier/Stage2/Stage2_grayV2.pt",
        "build": lambda: SimpleCNN_GAP2(in_channels=1, num_classes=2),
        "class_names": {0: "Overlap", 1: "Isolated"},
    },
    {
        "name": "Stage3",
        "ckpt": "../../CNN_models/classifier/Stage3/Stage3_gray2.pt",
        "build": lambda: SimpleCNN_GAP2(in_channels=1, num_classes=2),
        "class_names": {0: "Cluttered", 1: "Not-cluttered"},
    },
]


# ========================================
# HELPERS
# ========================================
def load_model(build_fn, ckpt_path: str):
    model = build_fn().to(DEVICE)

    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)

    if isinstance(ckpt, dict):
        if "model_state_dict" in ckpt:
            state = ckpt["model_state_dict"]
        elif "model_state" in ckpt:
            state = ckpt["model_state"]
        else:
            state = ckpt
    else:
        state = ckpt

    if any(k.startswith("module.") for k in state.keys()):
        state = {k.replace("module.", "", 1): v for k, v in state.items()}

    missing, unexpected = model.load_state_dict(state, strict=False)
    model.eval()
    return model


@torch.no_grad()
def predict_one(model, x, class_names):
    logits = model(x)
    probs = torch.softmax(logits, dim=1)

    pred_id = int(probs.argmax(dim=1).item())
    confidence = float(probs.max(dim=1).values.item())
    pred_name = class_names[pred_id]

    return pred_id, pred_name, confidence, probs.squeeze(0).cpu().tolist()


def format_class_mapping(class_names):
    return ", ".join([f"{idx}={name}" for idx, name in sorted(class_names.items())])


def make_black_border_white(pil_img, threshold=15):
    """
    Change near-black background/border pixels to white.
    Works for grayscale and RGB.
    """
    if IMAGE_MODE == "gray":
        img = pil_img.convert("L")
        arr = np.array(img)

        # pixels close to black become white
        border_mask = arr <= threshold
        arr[border_mask] = 255

        return Image.fromarray(arr).convert("L")

    else:
        img = pil_img.convert("RGB")
        arr = np.array(img)

        # near-black RGB pixels become white
        border_mask = np.all(arr <= threshold, axis=2)
        arr[border_mask] = [255, 255, 255]

        return Image.fromarray(arr).convert("RGB")

def crop_white_border(pil_img, threshold=252, margin=0):
    """
    Crop away near-white border around the real X-ray content.
    Higher threshold = more aggressive white border removal.
    """
    img = pil_img.convert("L")
    arr = np.array(img)

    # pixels darker than threshold are treated as real content
    mask = arr < threshold

    if not mask.any():
        return pil_img

    ys, xs = np.where(mask)

    left = max(xs.min() - margin, 0)
    right = min(xs.max() + margin, pil_img.size[0] - 1)
    top = max(ys.min() - margin, 0)
    bottom = min(ys.max() + margin, pil_img.size[1] - 1)

    return pil_img.crop((left, top, right + 1, bottom + 1))
    

def resize_to_1500_no_border(pil_img, target_size=1500):
    """
    Force resize image to exactly 1500x1500.
    No padding, no white border.
    This does NOT preserve aspect ratio.
    """
    return pil_img.resize((target_size, target_size), Image.BICUBIC)
    

def show_input_image(image_path):
    img_raw = Image.open(image_path)
    print(f"Original image mode: {img_raw.mode}")

    img = make_black_border_white(img_raw, threshold=15)
    img = crop_white_border(img, threshold=100, margin=0)
    img = resize_to_1500_no_border(img, target_size=1500)

    print("Final image size:", img.size)

    plt.figure(figsize=(6, 6))
    if IMAGE_MODE == "gray":
        plt.imshow(img, cmap="gray", vmin=0, vmax=255)
    else:
        plt.imshow(img)

    plt.axis("off")
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
    plt.margins(0)
    plt.show()

    return img


# ========================================
# RUN PREDICTION
# ========================================
img = show_input_image(IMAGE_PATH)
img_np = np.array(img)
print("Numpy shape:", img_np.shape)

if IMAGE_MODE == "gray":
    print("Top-left pixel gray:", img_np[0, 0])
else:
    print("Top-left pixel RGB:", img_np[0, 0])

x = transform(img).unsqueeze(0).to(DEVICE)
print("Input tensor shape:", x.shape)
print("Input tensor min/max:", x.min().item(), x.max().item())

print("\n===== PREDICTIONS =====")
prediction_results = []

for spec in MODEL_SPECS:
    model = load_model(spec["build"], spec["ckpt"])
    pred_id, pred_name, conf, probs = predict_one(model, x, spec["class_names"])

    prediction_results.append({
        "spec": spec,
        "model": model,
        "pred_id": pred_id,
        "pred_name": pred_name,
        "confidence": conf,
        "probs": probs,
    })

    print(f"\n{spec['name']}")
    print(f"  Class meaning: {format_class_mapping(spec['class_names'])}")
    print(f"  Predicted: {pred_name} (class {pred_id})")
    print(f"  Confidence: {conf:.4f}")
    print("  Per-class probabilities:")
    for class_id, class_name in sorted(spec["class_names"].items()):
        print(f"    class {class_id} = {class_name}: {probs[class_id]:.4f}")

print("\n=======================\n")

In [ ]:
# ========================================
# GRAD-CAM CONFIG
# ========================================
GRADCAM_ALPHA = 0.40
SAVE_GRADCAM = False
GRADCAM_SAVE_DIR = "./gradcam_outputs"
SHOW_ONLY_PREDICTED_CLASS = True


# ========================================
# GRAD-CAM HELPERS
# ========================================
def get_all_conv_layers(model: nn.Module):
    conv_layers = OrderedDict()
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            conv_layers[name] = module

    if not conv_layers:
        raise RuntimeError("No Conv2d layers found in model")

    return conv_layers


def unnormalize_imagenet_tensor(x: torch.Tensor):
    img = x.detach().cpu().squeeze(0).permute(1, 2, 0).numpy().copy()

    if IMAGE_MODE == "rgb":
        mean = np.array(IMAGENET_MEAN, dtype=np.float32)
        std = np.array(IMAGENET_STD, dtype=np.float32)
        img = img * std + mean

    img = np.clip(img, 0, 1)
    return img


def make_heatmap_rgb(cam_01: np.ndarray):
    heatmap_u8 = np.uint8(np.clip(cam_01, 0, 1) * 255)
    heatmap_bgr = cv2.applyColorMap(heatmap_u8, cv2.COLORMAP_JET)
    heatmap_rgb = cv2.cvtColor(heatmap_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    return heatmap_rgb


def overlay_heatmap(base_img_01: np.ndarray, heatmap_rgb_01: np.ndarray, alpha: float = 0.40):
    overlay = (1 - alpha) * base_img_01 + alpha * heatmap_rgb_01
    return np.clip(overlay, 0, 1)


def compute_gradcam_for_layer(model: nn.Module, x: torch.Tensor, target_layer: nn.Module, target_class: int = None):
    model.eval()

    activations = []
    gradients = []

    def forward_hook(module, inp, out):
        activations.append(out.detach())

    def backward_hook(module, grad_input, grad_output):
        gradients.append(grad_output[0].detach())

    h1 = target_layer.register_forward_hook(forward_hook)
    h2 = target_layer.register_full_backward_hook(backward_hook)

    try:
        logits = model(x)
        probs = torch.softmax(logits, dim=1)
        pred_id = int(probs.argmax(dim=1).item())

        if target_class is None:
            target_class = pred_id

        score = logits[:, target_class].sum()

        model.zero_grad(set_to_none=True)
        score.backward()

        acts = activations[0]
        grads = gradients[0]

        weights = grads.mean(dim=(2, 3), keepdim=True)
        cam = (weights * acts).sum(dim=1, keepdim=True)
        cam = F.relu(cam)

        cam = cam.squeeze().cpu().numpy().astype(np.float32)

        if cam.max() > 0:
            cam = cam / (cam.max() + 1e-8)

        H, W = x.shape[2], x.shape[3]
        cam_resized = cv2.resize(cam, (W, H), interpolation=cv2.INTER_LINEAR)

        return cam_resized, pred_id, probs.squeeze(0).detach().cpu().numpy()

    finally:
        h1.remove()
        h2.remove()


def compute_gradcam_all_layers(model: nn.Module, x: torch.Tensor, target_class: int = None):
    conv_layers = get_all_conv_layers(model)
    results = []

    for layer_name, layer_module in conv_layers.items():
        cam, pred_id, probs_np = compute_gradcam_for_layer(
            model=model,
            x=x,
            target_layer=layer_module,
            target_class=target_class,
        )

        results.append({
            "layer_name": layer_name,
            "cam": cam,
            "pred_id": pred_id,
            "probs": probs_np,
        })

    return results


def show_gradcam_grid(model_name, base_img_01, gradcam_results, class_names, confidence):
    n = len(gradcam_results)
    cols = 3
    rows = int(np.ceil(n / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = np.array(axes).reshape(-1)

    for ax, result in zip(axes, gradcam_results):
        layer_name = result["layer_name"]
        cam_01 = result["cam"]
        pred_id = result["pred_id"]
        pred_name = class_names[pred_id]

        heatmap_rgb = make_heatmap_rgb(cam_01)
        overlay = overlay_heatmap(base_img_01, heatmap_rgb, alpha=GRADCAM_ALPHA)

        ax.imshow(overlay)
        ax.set_title(f"{model_name}\n{layer_name}\nPred: {pred_name} | conf={confidence:.4f}")
        ax.axis("off")

    for ax in axes[len(gradcam_results):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


def show_gradcam_detailed_triplets(model_name, base_img_01, gradcam_results, class_names, confidence):
    for result in gradcam_results:
        layer_name = result["layer_name"]
        cam_01 = result["cam"]
        pred_id = result["pred_id"]
        pred_name = class_names[pred_id]

        heatmap_rgb = make_heatmap_rgb(cam_01)
        overlay = overlay_heatmap(base_img_01, heatmap_rgb, alpha=GRADCAM_ALPHA)

        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        axes[0].imshow(base_img_01)
        axes[0].set_title(f"{model_name}\nInput")
        axes[0].axis("off")

        axes[1].imshow(heatmap_rgb)
        axes[1].set_title(f"{layer_name}\nHeatmap")
        axes[1].axis("off")

        axes[2].imshow(overlay)
        axes[2].set_title(f"{layer_name}\nOverlay\nPred: {pred_name} | conf={confidence:.4f}")
        axes[2].axis("off")

        plt.tight_layout()
        plt.show()


def save_gradcam_triplets(model_name, base_img_01, gradcam_results, class_names, confidence, save_root):
    os.makedirs(save_root, exist_ok=True)

    for result in gradcam_results:
        layer_name = result["layer_name"].replace(".", "_")
        cam_01 = result["cam"]
        pred_id = result["pred_id"]
        pred_name = class_names[pred_id]

        heatmap_rgb = make_heatmap_rgb(cam_01)
        overlay = overlay_heatmap(base_img_01, heatmap_rgb, alpha=GRADCAM_ALPHA)

        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        axes[0].imshow(base_img_01)
        axes[0].set_title("Input")
        axes[0].axis("off")

        axes[1].imshow(heatmap_rgb)
        axes[1].set_title(f"{layer_name} heatmap")
        axes[1].axis("off")

        axes[2].imshow(overlay)
        axes[2].set_title(f"{pred_name} | conf={confidence:.4f}")
        axes[2].axis("off")

        plt.tight_layout()
        out_path = os.path.join(save_root, f"{model_name}_{layer_name}_gradcam.png")
        plt.savefig(out_path, dpi=200, bbox_inches="tight")
        plt.close(fig)


# ========================================
# RUN GRAD-CAM
# ========================================
base_img_01 = unnormalize_imagenet_tensor(x)

print("\n===== DETAILED PER-LAYER GRAD-CAM =====")

for result in prediction_results:
    spec = result["spec"]
    model = result["model"]
    pred_id = result["pred_id"]
    conf = result["confidence"]

    print(f"\n{spec['name']}")
    conv_layers = get_all_conv_layers(model)
    print("  Conv layers used for Grad-CAM:")
    for layer_name in conv_layers.keys():
        print(f"    - {layer_name}")

    target_class = pred_id if SHOW_ONLY_PREDICTED_CLASS else None

    gradcam_results = compute_gradcam_all_layers(
        model=model,
        x=x,
        target_class=target_class,
    )

    print(f"  Generated Grad-CAM for {len(gradcam_results)} conv layers")

    show_gradcam_grid(
        model_name=spec["name"],
        base_img_01=base_img_01,
        gradcam_results=gradcam_results,
        class_names=spec["class_names"],
        confidence=conf,
    )

    show_gradcam_detailed_triplets(
        model_name=spec["name"],
        base_img_01=base_img_01,
        gradcam_results=gradcam_results,
        class_names=spec["class_names"],
        confidence=conf,
    )

    if SAVE_GRADCAM:
        save_dir = os.path.join(GRADCAM_SAVE_DIR, spec["name"])
        save_gradcam_triplets(
            model_name=spec["name"],
            base_img_01=base_img_01,
            gradcam_results=gradcam_results,
            class_names=spec["class_names"],
            confidence=conf,
            save_root=save_dir,
        )
        print(f"  Saved Grad-CAM images to: {save_dir}")

print("\n=======================================\n")